# Aula 17 - Notebook: Problema do Caixeiro-Viajante (TSP) e Roteamento Logístico de AGVs

Neste notebook implementamos a heurística do **Vizinho Mais Próximo** e a busca local **2-Opt** para otimizar o percurso de amostragem do AGV industrial.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

from typing import List

class RoteadorAGV_TSP:
    def __init__(self, locais: List[str], matriz_distancias: List[List[float]]):
        self.locais = locais
        self.dist = matriz_distancias
        self.n = len(locais)
        
    def calcular_custo_rota(self, rota: List[int]) -> float:
        c = sum(self.dist[rota[i]][rota[i+1]] for i in range(len(rota)-1))
        c += self.dist[rota[-1]][rota[0]]
        return c

    def vizinho_mais_proximo(self, inicio_idx: int = 0) -> List[int]:
        nao_vis = set(range(self.n))
        nao_vis.remove(inicio_idx)
        rota = [inicio_idx]
        atual = inicio_idx
        while nao_vis:
            prox = min(nao_vis, key=lambda v: self.dist[atual][v])
            rota.append(prox)
            nao_vis.remove(prox)
            atual = prox
        return rota

    def otimizar_2opt(self, rota: List[int]) -> List[int]:
        melhor = list(rota)
        melhor_c = self.calcular_custo_rota(melhor)
        melhorou = True
        while melhorou:
            melhorou = False
            for i in range(1, self.n - 1):
                for j in range(i + 1, self.n):
                    nova = melhor[:i] + melhor[i:j+1][::-1] + melhor[j+1:]
                    c = self.calcular_custo_rota(nova)
                    if c < melhor_c:
                        melhor_c = c
                        melhor = nova
                        melhorou = True
                        break
                if melhorou: break
        return melhor

locais = ["Lab", "TK301", "TK302", "R101", "TK303", "GRAN201"]
dist_mat = [
    [0.0, 45.0, 50.0, 80.0, 75.0, 110.0],
    [45.0, 0.0, 20.0, 55.0, 60.0, 95.0],
    [50.0, 20.0, 0.0, 45.0, 50.0, 85.0],
    [80.0, 55.0, 45.0, 0.0, 25.0, 40.0],
    [75.0, 60.0, 50.0, 25.0, 0.0, 35.0],
    [110.0, 95.0, 85.0, 40.0, 35.0, 0.0]
]

agv_tsp = RoteadorAGV_TSP(locais, dist_mat)
r_ini = agv_tsp.vizinho_mais_proximo(0)
r_opt = agv_tsp.otimizar_2opt(r_ini)

print("Rota Inicial NN:", " -> ".join([locais[i] for i in r_ini]) + f" | Distância: {agv_tsp.calcular_custo_rota(r_ini):.1f}m")
print("Rota Ótima 2-Opt:", " -> ".join([locais[i] for i in r_opt]) + f" | Distância: {agv_tsp.calcular_custo_rota(r_opt):.1f}m")


Rota Inicial NN: Lab -> TK301 -> TK302 -> R101 -> TK303 -> GRAN201 | Distância: 280.0m
Rota Ótima 2-Opt: Lab -> TK301 -> TK302 -> R101 -> GRAN201 -> TK303 | Distância: 260.0m
